# Week 9: Your Own Drawings: Building an Image Dataset

**Grade band:** 6 to 8  |  **Duration:** 60 minutes  |  **Platform:** JupyterLite (browser, no account) or Google Colab

**How to use this notebook:** run each cell from top to bottom with Shift + Enter. Read the text, run the code, then complete the challenge cells marked **SOLUTION**. Save your work at the end of the session (File > Download) so it can be uploaded to your portfolio.

> **Teacher copy.** This notebook contains completed answers for every tier. Do not distribute to students. Use it to check work against the validation checklist.

## Hook: Can it read YOUR handwriting?

Last week the model read handwriting it had seen before. Today you draw a digit yourself, turn it into 64 numbers, and find out if the model can read it. Then you build a dataset of shapes from scratch and train a brand new classifier on it.

In [ ]:
# Setup: run this cell first.
import pandas as pd
import matplotlib.pyplot as plt

# If a data file is not found next to this notebook (for example on Google Colab),
# it is loaded from the Wize data folder online instead. Replace this URL after publishing.
DATA_URL = "https://raw.githubusercontent.com/wizeacademy/ml-ai-6-8/main/notebooks/data/"

def load(name):
    """Load a Wize dataset by file name, from the local data folder or from the web."""
    try:
        return pd.read_csv("data/" + name)
    except Exception:
        return pd.read_csv(DATA_URL + name)

print("Setup complete. pandas and matplotlib are ready.")

## Teach 1: Train the digit reader again

Same recipe as week 8: load the digits, split, fit K nearest neighbors. Two lines you already know.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(digits.data, digits.target, test_size=0.25, random_state=42)
digit_model = KNeighborsClassifier(n_neighbors=3)
digit_model.fit(X_train, y_train)
print("Accuracy on hidden digits:", round(digit_model.score(X_test, y_test) * 100, 1), "%")

## Teach 2: Turning any picture into the 64 numbers the model expects

A photo or drawing is thousands of pixels in color. The model was trained on 8 by 8 grayscale pixels scored 0 to 16, with ink as **high** numbers. The `Pillow` library converts any image with four steps: grayscale, resize to 8 by 8, invert (so ink is high), and scale to 0 to 16.

**Concept checkpoint:** why must we resize to exactly 8 by 8? (Hint: how many features did the model train on?)

In [ ]:
from PIL import Image, ImageDraw, ImageOps

def image_to_features(img):
    """Convert a Pillow image into the 64 numbers a digits model expects."""
    img = img.convert("L")                       # grayscale
    img = img.resize((8, 8), Image.LANCZOS)      # 8 by 8 pixels
    img = ImageOps.invert(img)                   # ink becomes high values
    pixels = np.array(img, dtype=float)          # 8 by 8 grid of 0 to 255
    pixels = pixels / 255 * 16                    # scale to 0 to 16 like the training data
    return pixels.reshape(1, 64), pixels

# Make a sample drawing in code so everyone has one to test with
sample = Image.new("L", (200, 200), color=255)
draw = ImageDraw.Draw(sample)
draw.line([(60, 40), (140, 40), (100, 40), (100, 160)], fill=0, width=22)   # a thick "7" shape
draw.line([(140, 40), (90, 160)], fill=0, width=22)

features, grid = image_to_features(sample)
print("Model says:", digit_model.predict(features)[0])

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(sample, cmap="gray"); axes[0].set_title("Your drawing"); axes[0].axis("off")
axes[1].imshow(grid, cmap="gray_r"); axes[1].set_title("What the model sees"); axes[1].axis("off")
plt.show()

## Teach 3: Loading a real file

Draw a single digit in any paint app (thick black marker on white), save it as `my_digit.png`, and upload it next to this notebook. Then run the cell below. If the file is missing, the cell tells you and moves on.

In [ ]:
import os
if os.path.exists("my_digit.png"):
    mine = Image.open("my_digit.png")
    features, grid = image_to_features(mine)
    print("Model says your digit is:", digit_model.predict(features)[0])
    plt.imshow(grid, cmap="gray_r"); plt.axis("off"); plt.show()
else:
    print("No my_digit.png found yet. Draw one, upload it, and run this cell again.")

## SOLUTION: Challenge

**Timer suggestion: 25 minutes.**

### Mild
Change the sample drawing so it looks like a different digit (edit the `draw.line` coordinates or add a `draw.ellipse`). Does the model read it correctly? Try two digits.

### Medium
Draw three digits in a paint app, upload them, and test all three. Record which ones the model gets right. For a miss, look at the 8 by 8 view and explain why the model was fooled (too thin? off center? too small?).

### Spicy
Build your **own image dataset**: generate 40 circles and 40 crosses with `ImageDraw` at random sizes and positions, convert each to 64 features, train a new K nearest neighbors model, and test it on a hand drawn shape.

In [ ]:
# MILD: draw a different digit and test it
canvas = Image.new("L", (200, 200), color=255)
draw = ImageDraw.Draw(canvas)
draw.ellipse([50, 30, 150, 170], outline=0, width=22)      # a "0"
features, grid = image_to_features(canvas)
print("Model says:", digit_model.predict(features)[0])
plt.imshow(grid, cmap="gray_r"); plt.axis("off"); plt.show()

canvas = Image.new("L", (200, 200), color=255)
draw = ImageDraw.Draw(canvas)
draw.line([(100, 30), (100, 170)], fill=0, width=22)        # a "1"
features, grid = image_to_features(canvas)
print("Model says:", digit_model.predict(features)[0])

In [ ]:
# MEDIUM: test three uploaded drawings (falls back to generated digits if no uploads)
my_files = ["digit_a.png", "digit_b.png", "digit_c.png"]
found = [f for f in my_files if os.path.exists(f)]
if not found:
    print("No uploads found, generating three test digits instead.")
    for i, coords in enumerate([[(100, 30), (100, 170)], [(60, 40), (140, 40), (100, 40), (100, 160)], [(60, 170), (140, 170), (100, 170), (100, 30)]]):
        c = Image.new("L", (200, 200), color=255); ImageDraw.Draw(c).line(coords, fill=0, width=22)
        c.save(f"digit_{'abc'[i]}.png")
    found = my_files
for name in found:
    features, grid = image_to_features(Image.open(name))
    print(name, "->", digit_model.predict(features)[0])
    plt.imshow(grid, cmap="gray_r"); plt.title(name); plt.axis("off"); plt.show()
# Typical miss explanation: a thin or off center stroke shrinks to a faint blur at 8 by 8,
# so the model matches it to the nearest blurry training digit, often a 1 or a 7.

In [ ]:
# SPICY: build a circle vs cross dataset and train on it
import random
random.seed(3)

def make_shape(kind):
    img = Image.new("L", (100, 100), color=255)
    d = ImageDraw.Draw(img)
    size = random.randint(35, 70)
    x = random.randint(5, 95 - size); y = random.randint(5, 95 - size)
    if kind == "circle":
        d.ellipse([x, y, x + size, y + size], outline=0, width=8)
    else:
        d.line([(x, y), (x + size, y + size)], fill=0, width=8)
        d.line([(x + size, y), (x, y + size)], fill=0, width=8)
    return img

X_shapes, y_shapes = [], []
for kind in ["circle", "cross"]:
    for _ in range(40):
        f, _grid = image_to_features(make_shape(kind))
        X_shapes.append(f[0]); y_shapes.append(kind)

Xs_train, Xs_test, ys_train, ys_test = train_test_split(X_shapes, y_shapes, test_size=0.25, random_state=1)
shape_model = KNeighborsClassifier(n_neighbors=3).fit(Xs_train, ys_train)
print("Shape classifier accuracy:", round(shape_model.score(Xs_test, ys_test) * 100, 1), "%")

test = make_shape("cross")
f, grid = image_to_features(test)
print("New shape ->", shape_model.predict(f)[0])
plt.imshow(grid, cmap="gray_r"); plt.axis("off"); plt.show()

## Extra activities (if you finish early)

- Add a third shape (triangle) to the Spicy dataset. Does accuracy drop? Why might it?
- Try `n_neighbors=1` and `n_neighbors=15` on the digit model. Which is better on your own drawings?
- Draw a digit with a very thin pen. Watch what happens at 8 by 8. Connect this to why apps ask you to "write clearly."

## Reflection

- What are the four steps that turn a picture into features?
- You built a dataset today. What would make it more **representative** of real drawings?